# DrugScope AI — Task 3: Exploratory Data Analysis
## Notebook 01 — Step 1: Dataset overview · Step 2: Data dictionary
**Mini Project in Machine Learning · kinase inhibitors from ChEMBL (raw data, 5 files)**

**Step 1 — Dataset overview.** Objective: understand the general structure of each dataset. For each of the five datasets: `shape`, `head()` / `tail()`, `sample()`, `columns`, `info()`, `describe()` and `describe(include="all")`.

**Step 2 — Understand the columns (data dictionary).** Objective: understand the meaning and purpose of every column: name, meaning, pandas data type, semantic type, encoding, missing values (number and %) and role in DrugScope AI. The result is one data dictionary per dataset, saved as a CSV file.

This notebook **only observes**: no dataset is cleaned or modified (the only files written are the dictionaries, which are documentation).

| Dataset | File | One row is… |
|---|---|---|
| molecules | `molecules_raw.csv` | one ChEMBL compound (**the clustering input**) |
| activities | `activities_raw.csv` | one activity measurement of a molecule on a kinase |
| indications | `indications_raw.csv` | one molecule–disease pair |
| mechanisms | `mechanisms_raw.csv` | one molecule–target mechanism record |
| targets | `targets_raw.csv` | one kinase protein |

## 0. Setup
We import the libraries and load the five raw CSV files into DataFrames: `mol`, `act`, `ind`, `mech`, `tgt`. The helper `describe_numeric()` is needed because `describe()` fails on a table without numeric columns (this is the case for `targets`).

In [3]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from eda_utils import load_raw, RAW

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 45)

mol, act, ind, mech, tgt = load_raw()
files = {"molecules": mol, "activities": act, "indications": ind, "mechanisms": mech, "targets": tgt}

def describe_numeric(df):
    num = df.select_dtypes("number")
    if num.shape[1] == 0:
        print("No numeric column: describe() has nothing to describe.")
        return None
    return num.describe().T.round(2)

print(RAW, "|", {k: v.shape for k, v in files.items()})

../data/raw | {'molecules': (407, 15), 'activities': (15655, 11), 'indications': (2864, 4), 'mechanisms': (610, 5), 'targets': (548, 4)}


**Conclusion:** the five files load without error, with the shapes announced in the data-collection document: molecules (407, 15), activities (15 655, 11), indications (2 864, 4), mechanisms (610, 5), targets (548, 4).

---
# Step 1 — Dataset overview
For every dataset we run the same seven checks, in this order: dimensions and column names, first and last rows, random rows, data types and non-null counts (`info()`), numerical statistics (`describe()`), and statistics for all columns (`describe(include="all")`).

---
## 1.1 Dataset: `molecules` — one row per ChEMBL compound; the only table used to build the clusters

**1.1a — Dimensions and column names.** `shape` gives the number of rows and columns; `columns` lists the variable names.

In [ ]:
print("shape:", mol.shape, "->", mol.shape[0], "rows x", mol.shape[1], "columns")
print("columns:", list(mol.columns))

shape: (407, 15) -> 407 rows x 15 columns
columns: ['molecule_chembl_id', 'name', 'max_phase', 'first_approval', 'molecule_type', 'smiles', 'inchi_key', 'mol_weight', 'alogp', 'hbd', 'hba', 'psa', 'rtb', 'aromatic_rings', 'ro5_violations']


**1.1b — First and last rows.** `head()` and `tail()` show how the data looks and whether the beginning and the end are different (for example sorted).

In [ ]:
print("head()"); display(mol.head())
print("tail()"); display(mol.tail())

head()


,molecule_chembl_id,name,max_phase,first_approval,molecule_type,smiles,inchi_key,mol_weight,alogp,hbd,hba,psa,rtb,aromatic_rings,ro5_violations
0,CHEMBL9960,AZD-7624,2.0,NaN,Small molecule,CC[C@H](NC(=O)c1cc(-c2ccccc2)nc2ccccc12)c...,MXNGYQJJYRVGGJ-QFIPXVFZSA-N,366.46,5.78,1.0,2.0,41.99,5.0,4.0,1.0
1,CHEMBL288441,BOSUTINIB,4.0,2012.0,Small molecule,COc1cc(Nc2c(C#N)cnc3cc(OCCCN4CCN(C)CC4)c(...,UBPYILGKFZZVDX-UHFFFAOYSA-N,530.46,5.19,1.0,8.0,82.88,9.0,3.0,2.0
2,CHEMBL103667,DORAMAPIMOD,2.0,NaN,Small molecule,Cc1ccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(OCC...,MVCOAUNKQVWQHZ-UHFFFAOYSA-N,527.67,5.99,2.0,6.0,80.65,7.0,4.0,2.0
3,CHEMBL1421,DASATINIB ANHYDROUS,4.0,2006.0,Small molecule,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2...,ZBNZXTGUTAYRHI-UHFFFAOYSA-N,488.02,3.31,3.0,9.0,106.51,7.0,3.0,0.0
4,CHEMBL1642,IMATINIB MESYLATE,4.0,2001.0,Small molecule,CS(=O)(=O)O.Cc1ccc(NC(=O)c2ccc(CN3CCN(C)C...,YLMAHDNUQAMNNX-UHFFFAOYSA-N,589.72,4.59,2.0,7.0,86.28,7.0,4.0,0.0


tail()


,molecule_chembl_id,name,max_phase,first_approval,molecule_type,smiles,inchi_key,mol_weight,alogp,hbd,hba,psa,rtb,aromatic_rings,ro5_violations
402,CHEMBL4297701,BIO-11006,2.0,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
403,CHEMBL4297728,CIRMTUZUMAB,3.0,NaN,Antibody,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404,CHEMBL4298013,ELGEMTUMAB,2.0,NaN,Antibody,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405,CHEMBL4650292,CDX-3379,2.0,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406,CHEMBL6067997,APILIMOD MESYLATE,2.0,NaN,Small molecule,COc1ccc(-c2nc(S(C)(=O)=O)[nH]c2-c2ccc(OC)...,QAFWUMUPMXXFFK-UHFFFAOYSA-N,550.63,3.16,1.0,5.0,81.28,5.0,3.0,0.0


**1.1c — Random rows.** `sample()` avoids the bias of looking only at the first and last rows (`random_state=42` makes it reproducible).

In [ ]:
mol.sample(5, random_state=42)

,molecule_chembl_id,name,max_phase,first_approval,molecule_type,smiles,inchi_key,mol_weight,alogp,hbd,hba,psa,rtb,aromatic_rings,ro5_violations
70,CHEMBL1088751,PH-797804,2.0,NaN,Small molecule,CNC(=O)c1ccc(C)c(-n2c(C)cc(OCc3ccc(F)cc3F...,KCAJXIDMCNPGHZ-UHFFFAOYSA-N,477.31,4.43,1.0,4.0,60.33,5.0,3.0,0.0
218,CHEMBL4742157,GSK-2256098,2.0,NaN,Small molecule,CONC(=O)c1ccccc1Nc1cc(Nc2cc(C)nn2C(C)C)nc...,BVAHPPKGOOJSPU-UHFFFAOYSA-N,414.90,4.60,3.0,7.0,93.10,7.0,3.0,0.0
258,CHEMBL4650215,INAVOLISIB,4.0,2024.0,Small molecule,C[C@H](Nc1ccc2c(c1)OCCn1cc(N3C(=O)OC[C@H]...,SGEUNORSOZVTOL-CABZTGNLSA-N,407.38,1.82,2.0,7.0,111.71,5.0,2.0,0.0
33,CHEMBL3545074,LEO-15520,1.0,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,CHEMBL495727,AT-9283,2.0,NaN,Small molecule,O=C(Nc1c[nH]nc1-c1nc2ccc(CN3CCOCC3)cc2[nH...,LOLPPWBBNUVNQZ-UHFFFAOYSA-N,381.44,2.07,4.0,5.0,110.96,5.0,3.0,0.0


**1.1d — Data types and non-null counts.** `info()` gives the pandas type of every column and how many values are present.

In [ ]:
mol.info()

<class 'pandas.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   molecule_chembl_id  407 non-null    str    
 1   name                407 non-null    str    
 2   max_phase           407 non-null    float64
 3   first_approval      88 non-null     float64
 4   molecule_type       407 non-null    str    
 5   smiles              345 non-null    str    
 6   inchi_key           345 non-null    str    
 7   mol_weight          345 non-null    float64
 8   alogp               343 non-null    float64
 9   hbd                 343 non-null    float64
 10  hba                 343 non-null    float64
 11  psa                 343 non-null    float64
 12  rtb                 343 non-null    float64
 13  aromatic_rings      343 non-null    float64
 14  ro5_violations      343 non-null    float64
dtypes: float64(10), str(5)
memory usage: 47.8 KB


**1.1e — Numerical statistics.** `describe()` summarizes the numeric columns: count, mean, standard deviation, minimum, quartiles and maximum (transposed so that each row is a column).

In [ ]:
describe_numeric(mol)

,count,mean,std,min,25%,50%,75%,max
max_phase,407.0,2.29,1.09,1.00,1.00,2.00,3.00,4.00
first_approval,88.0,2016.25,7.54,1974.00,2013.75,2018.00,2020.25,2024.00
mol_weight,345.0,503.35,280.29,236.36,412.42,468.48,540.02,3823.50
alogp,343.0,3.83,1.48,-2.87,2.88,3.69,4.88,11.17
hbd,343.0,2.15,1.18,0.00,1.50,2.00,3.00,14.00
hba,343.0,6.76,2.10,1.00,5.00,7.00,8.00,14.00
psa,343.0,97.94,31.17,12.03,80.55,94.14,110.66,355.24
rtb,343.0,5.82,2.60,0.00,4.00,6.00,7.00,22.00
aromatic_rings,343.0,3.38,1.06,0.00,3.00,3.00,4.00,8.00
ro5_violations,343.0,0.58,0.77,0.00,0.00,0.00,1.00,3.00


**1.1f — Statistics for all columns.** `describe(include="all")` adds the text columns: number of distinct values (`unique`), most frequent value (`top`) and its frequency (`freq`).

In [ ]:
mol.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
molecule_chembl_id,407,407,CHEMBL9960,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,407,407,AZD-7624,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max_phase,407.0,NaN,NaN,NaN,2.29484,1.090283,1.0,1.0,2.0,3.0,4.0
first_approval,88.0,NaN,NaN,NaN,2016.25,7.541837,1974.0,2013.75,2018.0,2020.25,2024.0
molecule_type,407,4,Small molecule,343,NaN,NaN,NaN,NaN,NaN,NaN,NaN
smiles,345,345,CC[C@H](NC(=O)c1cc(-c2ccccc2)nc2ccccc12)c...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
inchi_key,345,345,MXNGYQJJYRVGGJ-QFIPXVFZSA-N,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mol_weight,345.0,NaN,NaN,NaN,503.350464,280.286931,236.36,412.42,468.48,540.02,3823.5
alogp,343.0,NaN,NaN,NaN,3.830583,1.482805,-2.87,2.875,3.69,4.88,11.17
hbd,343.0,NaN,NaN,NaN,2.151603,1.179896,0.0,1.5,2.0,3.0,14.0


**Conclusion 1.1 — molecules**
- **Size and types:** 407 rows (one per compound) × 15 columns: 10 numeric (`float64`) and 5 text columns (`object`, displayed as `str` in pandas ≥ 3).
- **`info()` reveals gaps:** `smiles`, `inchi_key` and `mol_weight` have 345 non-null values, the other descriptors 343, and `first_approval` only 88. All the other columns are complete. The structure-related columns are missing together (they are analyzed in the data-quality step).
- **`describe()`:** `max_phase` goes from 1 to 4 (mean 2.29) and the approval years from 1974 to 2024 (median 2018). `mol_weight` has a mean of 503 but a maximum of 3 823 while 75% of the molecules are below 540: a couple of non-small molecules (peptides) stretch it. The other descriptors look plausible (logP from −2.9 to 11.2, TPSA from 12 to 355, at most 14 H-bond donors).
- **`describe(include="all")`:** `molecule_chembl_id` and `name` are unique (407 distinct values); `molecule_type` has 4 categories, the most frequent being *Small molecule* (343); `smiles` and `inchi_key` have 345 distinct values among their 345 non-missing entries.

---
## 1.2 Dataset: `activities` — one row per activity measurement (the largest table)

**1.2a — Dimensions and column names.** `shape` gives the number of rows and columns; `columns` lists the variable names.

In [ ]:
print("shape:", act.shape, "->", act.shape[0], "rows x", act.shape[1], "columns")
print("columns:", list(act.columns))

shape: (15655, 11) -> 15655 rows x 11 columns
columns: ['assay_chembl_id', 'molecule_chembl_id', 'pchembl_value', 'standard_type', 'standard_units', 'standard_value', 'target_chembl_id', 'target_pref_name', 'type', 'units', 'value']


**1.2b — First and last rows.** `head()` and `tail()` show how the data looks and whether the beginning and the end are different (for example sorted).

In [ ]:
print("head()"); display(act.head())
print("tail()"); display(act.tail())

head()


,assay_chembl_id,molecule_chembl_id,pchembl_value,standard_type,standard_units,standard_value,target_chembl_id,target_pref_name,type,units,value
0,CHEMBL882727,CHEMBL288441,8.89,IC50,nM,1.3,CHEMBL267,Proto-oncogene tyrosine-protein kinase Src,IC50,nM,1.3
1,CHEMBL806435,CHEMBL288441,8.92,IC50,nM,1.2,CHEMBL267,Proto-oncogene tyrosine-protein kinase Src,IC50,nM,1.2
2,CHEMBL806425,CHEMBL288441,8.92,IC50,nM,1.2,CHEMBL267,Proto-oncogene tyrosine-protein kinase Src,IC50,nM,1.2
3,CHEMBL806424,CHEMBL288441,7.00,IC50,nM,100.0,CHEMBL267,Proto-oncogene tyrosine-protein kinase Src,IC50,nM,100.0
4,CHEMBL818278,CHEMBL288441,8.92,IC50,nM,1.2,CHEMBL267,Proto-oncogene tyrosine-protein kinase Src,IC50,nM,1.2


tail()


,assay_chembl_id,molecule_chembl_id,pchembl_value,standard_type,standard_units,standard_value,target_chembl_id,target_pref_name,type,units,value
15650,CHEMBL5320561,CHEMBL4297643,7.85,IC50,nM,14.00,CHEMBL1938222,1-phosphatidylinositol 3-phosphate 5-kinase,IC50,nM,14.00
15651,CHEMBL5324473,CHEMBL4297643,7.85,IC50,nM,14.00,CHEMBL1938222,1-phosphatidylinositol 3-phosphate 5-kinase,IC50,nM,14.00
15652,CHEMBL5506169,CHEMBL4297643,7.85,IC50,nM,14.00,CHEMBL1938222,1-phosphatidylinositol 3-phosphate 5-kinase,IC50,nM,14.00
15653,CHEMBL6101025,CHEMBL4297643,8.47,IC50,nM,3.40,CHEMBL1938222,1-phosphatidylinositol 3-phosphate 5-kinase,IC50,nM,3.40
15654,CHEMBL6101026,CHEMBL4297643,9.51,IC50,nM,0.31,CHEMBL1938222,1-phosphatidylinositol 3-phosphate 5-kinase,IC50,nM,0.31


**1.2c — Random rows.** `sample()` avoids the bias of looking only at the first and last rows (`random_state=42` makes it reproducible).

In [ ]:
act.sample(5, random_state=42)

,assay_chembl_id,molecule_chembl_id,pchembl_value,standard_type,standard_units,standard_value,target_chembl_id,target_pref_name,type,units,value
2009,CHEMBL1908504,CHEMBL180022,6.17,Kd,nM,680.0,CHEMBL4599,Activated CDC42 kinase 1,Kd,nM,680.000
169,CHEMBL1261656,CHEMBL103667,7.66,Ki,nM,22.0,CHEMBL260,Mitogen-activated protein kinase 14,Ki,uM,0.022
1402,CHEMBL3991772,CHEMBL206834,6.27,Kd,nM,536.0,CHEMBL260,Mitogen-activated protein kinase 14,Kd apparent,nM,536.000
2126,CHEMBL3991638,CHEMBL217092,4.99,Kd,nM,10153.0,CHEMBL5251,Tyrosine-protein kinase BTK,Kd apparent,nM,10153.000
11274,CHEMBL1908585,CHEMBL535,5.00,Kd,nM,9900.0,CHEMBL5332,Mitogen-activated protein kinase 7,Kd,nM,9900.000


**1.2d — Data types and non-null counts.** `info()` gives the pandas type of every column and how many values are present.

In [ ]:
act.info()

<class 'pandas.DataFrame'>
RangeIndex: 15655 entries, 0 to 15654
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   assay_chembl_id     15655 non-null  str    
 1   molecule_chembl_id  15655 non-null  str    
 2   pchembl_value       15655 non-null  float64
 3   standard_type       15655 non-null  str    
 4   standard_units      15655 non-null  str    
 5   standard_value      15655 non-null  float64
 6   target_chembl_id    15655 non-null  str    
 7   target_pref_name    15655 non-null  str    
 8   type                15655 non-null  str    
 9   units               14980 non-null  str    
 10  value               15655 non-null  float64
dtypes: float64(3), str(8)
memory usage: 1.3 MB


**1.2e — Numerical statistics.** `describe()` summarizes the numeric columns: count, mean, standard deviation, minimum, quartiles and maximum (transposed so that each row is a column).

In [ ]:
describe_numeric(act)

,count,mean,std,min,25%,50%,75%,max
pchembl_value,15655.0,6.86,1.19,4.00,5.91,6.7,7.72,10.8
standard_value,15655.0,1415.30,4295.88,0.02,19.00,200.0,1233.00,100000.0
value,15655.0,1161.01,3640.77,0.00,6.80,102.0,970.00,99233.0


**1.2f — Statistics for all columns.** `describe(include="all")` adds the text columns: number of distinct values (`unique`), most frequent value (`top`) and its frequency (`freq`).

In [ ]:
act.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
assay_chembl_id,15655,6679,CHEMBL3991867,47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
molecule_chembl_id,15655,262,CHEMBL535,918,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pchembl_value,15655.0,NaN,NaN,NaN,6.862639,1.187401,4.0,5.91,6.7,7.72,10.8
standard_type,15655,3,Kd,9093,NaN,NaN,NaN,NaN,NaN,NaN,NaN
standard_units,15655,1,nM,15655,NaN,NaN,NaN,NaN,NaN,NaN,NaN
standard_value,15655.0,NaN,NaN,NaN,1415.303308,4295.87511,0.016,19.0,200.0,1233.0,100000.0
target_chembl_id,15655,381,CHEMBL1974,726,NaN,NaN,NaN,NaN,NaN,NaN,NaN
target_pref_name,15655,381,Receptor-type tyrosine-protein kinase FLT3,726,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,15655,11,Kd,6679,NaN,NaN,NaN,NaN,NaN,NaN,NaN
units,14980,15,nM,13678,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Conclusion 1.2 — activities**
- **Size and types:** 15 655 rows (one per measurement) × 11 columns: 3 numeric (`pchembl_value`, `standard_value`, `value`) and 8 text columns.
- **`info()`:** every column is complete except `units` (14 980 non-null, so 675 missing).
- **`describe()`:** `pchembl_value` goes from 4.0 to 10.8 (mean 6.86, median 6.7). `standard_value` goes from 0.02 to 100 000 nM with a median of 200 but a mean of 1 415 (std 4 296): it is extremely right-skewed and spans several orders of magnitude, so it should be analyzed on a log scale (which `pchembl_value` already is). The raw `value` (median 102, mean 1 161) mixes units.
- **`describe(include="all")`:** the table covers 262 molecules, 381 kinases and 6 679 assays. `standard_type` has 3 values (Kd is the most frequent with 9 093 rows, 58%) and `standard_units` is "nM" in every row, so it carries no information. Measurements are very unevenly spread: the most-measured molecule has 918 records and the most-measured kinase (FLT3) has 726.

---
## 1.3 Dataset: `indications` — one row per molecule–disease pair

**1.3a — Dimensions and column names.** `shape` gives the number of rows and columns; `columns` lists the variable names.

In [ ]:
print("shape:", ind.shape, "->", ind.shape[0], "rows x", ind.shape[1], "columns")
print("columns:", list(ind.columns))

shape: (2864, 4) -> 2864 rows x 4 columns
columns: ['efo_term', 'max_phase_for_ind', 'mesh_heading', 'molecule_chembl_id']


**1.3b — First and last rows.** `head()` and `tail()` show how the data looks and whether the beginning and the end are different (for example sorted).

In [ ]:
print("head()"); display(ind.head())
print("tail()"); display(ind.tail())

head()


,efo_term,max_phase_for_ind,mesh_heading,molecule_chembl_id
0,non-small cell lung carcinoma,4.0,"Carcinoma, Non-Small-Cell Lung",CHEMBL2105712
1,glioblastoma multiforme,2.0,Glioblastoma,CHEMBL288441
2,neoplasm,3.0,Neoplasms,CHEMBL288441
3,chronic lymphocytic leukemia,2.0,"Leukemia, Lymphocytic, Chronic, B-Cell",CHEMBL1421
4,leukemia,3.0,Leukemia,CHEMBL1421


tail()


,efo_term,max_phase_for_ind,mesh_heading,molecule_chembl_id
2859,triple-negative breast cancer,2.0,Triple Negative Breast Neoplasms,CHEMBL4297728
2860,gastric cancer,2.0,Stomach Neoplasms,CHEMBL4297728
2861,ovarian cancer,2.0,Ovarian Neoplasms,CHEMBL4297728
2862,prostate cancer,1.0,"Prostatic Neoplasms, Castration-Resistant",CHEMBL4297728
2863,urothelial carcinoma,1.0,Urinary Bladder Neoplasms,CHEMBL4297728


**1.3c — Random rows.** `sample()` avoids the bias of looking only at the first and last rows (`random_state=42` makes it reproducible).

In [ ]:
ind.sample(5, random_state=42)

,efo_term,max_phase_for_ind,mesh_heading,molecule_chembl_id
1590,endometrial neoplasm,2.0,Endometrial Neoplasms,CHEMBL3935857
1752,chronic obstructive pulmonary disease,2.0,"Pulmonary Disease, Chronic Obstructive",CHEMBL5314409
772,esophageal cancer,1.0,Esophageal Neoplasms,CHEMBL574737
1735,cancer,4.0,Neoplasms,CHEMBL2396661
387,gallbladder cancer,2.0,Gallbladder Neoplasms,CHEMBL1201179


**1.3d — Data types and non-null counts.** `info()` gives the pandas type of every column and how many values are present.

In [ ]:
ind.info()

<class 'pandas.DataFrame'>
RangeIndex: 2864 entries, 0 to 2863
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   efo_term            2864 non-null   str    
 1   max_phase_for_ind   2864 non-null   float64
 2   mesh_heading        2864 non-null   str    
 3   molecule_chembl_id  2864 non-null   str    
dtypes: float64(1), str(3)
memory usage: 89.6 KB


**1.3e — Numerical statistics.** `describe()` summarizes the numeric columns: count, mean, standard deviation, minimum, quartiles and maximum (transposed so that each row is a column).

In [ ]:
describe_numeric(ind)

,count,mean,std,min,25%,50%,75%,max
max_phase_for_ind,2864.0,1.91,0.9,0.5,1.0,2.0,2.0,4.0


**1.3f — Statistics for all columns.** `describe(include="all")` adds the text columns: number of distinct values (`unique`), most frequent value (`top`) and its frequency (`freq`).

In [ ]:
ind.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
efo_term,2864,495,neoplasm,178,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max_phase_for_ind,2864.0,NaN,NaN,NaN,1.91463,0.902393,0.5,1.0,2.0,2.0,4.0
mesh_heading,2864,362,Neoplasms,283,NaN,NaN,NaN,NaN,NaN,NaN,NaN
molecule_chembl_id,2864,359,CHEMBL1421,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Conclusion 1.3 — indications**
- **Size and types:** 2 864 rows (one per molecule–disease pair) × 4 columns: 1 numeric (`max_phase_for_ind`) and 3 text columns. No missing values.
- **`describe()`:** `max_phase_for_ind` goes from 0.5 to 4 (mean 1.91). It is an ordinal code, not a measurement, so its mean says little; the real distribution is 21 records at 0.5, 997 in phase 1, 1 324 in phase 2, 260 in phase 3 and 262 in phase 4 (approved).
- **`describe(include="all")`:** 359 distinct molecules, 495 distinct EFO terms and 362 distinct MeSH headings. The most frequent disease terms are generic cancer terms ("neoplasm" in 178 records, "Neoplasms" in 283), and one molecule alone has 88 indication records.

---
## 1.4 Dataset: `mechanisms` — one row per molecule–target mechanism record

**1.4a — Dimensions and column names.** `shape` gives the number of rows and columns; `columns` lists the variable names.

In [ ]:
print("shape:", mech.shape, "->", mech.shape[0], "rows x", mech.shape[1], "columns")
print("columns:", list(mech.columns))

shape: (610, 5) -> 610 rows x 5 columns
columns: ['action_type', 'max_phase', 'mechanism_of_action', 'molecule_chembl_id', 'target_chembl_id']


**1.4b — First and last rows.** `head()` and `tail()` show how the data looks and whether the beginning and the end are different (for example sorted).

In [ ]:
print("head()"); display(mech.head())
print("tail()"); display(mech.tail())

head()


,action_type,max_phase,mechanism_of_action,molecule_chembl_id,target_chembl_id
0,INHIBITOR,4,Receptor protein-tyrosine kinase erbB-2 i...,CHEMBL1201585,CHEMBL1824
1,INHIBITOR,4,Receptor protein-tyrosine kinase erbB-2 i...,CHEMBL2007641,CHEMBL1824
2,INHIBITOR,4,Receptor protein-tyrosine kinase erbB-2 i...,CHEMBL1201179,CHEMBL1824
3,INHIBITOR,4,Tyrosine-protein kinase ABL inhibitor,CHEMBL1201740,CHEMBL1862
4,INHIBITOR,4,Tyrosine-protein kinase ABL inhibitor,CHEMBL1421,CHEMBL1862


tail()


,action_type,max_phase,mechanism_of_action,molecule_chembl_id,target_chembl_id
605,INHIBITOR,3,Mitogen-activated protein kinase kinase k...,CHEMBL290352,CHEMBL1908389
606,INHIBITOR,2,1-phosphatidylinositol 3-phosphate 5-kina...,CHEMBL4297643,CHEMBL1938222
607,INHIBITOR,2,1-phosphatidylinositol 3-phosphate 5-kina...,CHEMBL6067997,CHEMBL1938222
608,INHIBITOR,3,Inactive tyrosine-protein kinase transmem...,CHEMBL4297728,CHEMBL4665585
609,INHIBITOR,2,Myristoylated alanine-rich C-kinase subst...,CHEMBL4297701,CHEMBL4804244


**1.4c — Random rows.** `sample()` avoids the bias of looking only at the first and last rows (`random_state=42` makes it reproducible).

In [ ]:
mech.sample(5, random_state=42)

,action_type,max_phase,mechanism_of_action,molecule_chembl_id,target_chembl_id
81,INHIBITOR,3,Tyrosine-protein kinase ABL inhibitor,CHEMBL3545413,CHEMBL1862
218,INHIBITOR,4,Tyrosine-protein kinase SYK inhibitor,CHEMBL3989516,CHEMBL2599
55,INHIBITOR,2,MAP kinase p38 alpha inhibitor,CHEMBL1230065,CHEMBL260
598,INHIBITOR,2,Receptor tyrosine-protein kinase erbB-3 i...,CHEMBL3707404,CHEMBL5838
264,INHIBITOR,1,PI3-kinase p110-alpha subunit inhibitor,CHEMBL1258517,CHEMBL4005


**1.4d — Data types and non-null counts.** `info()` gives the pandas type of every column and how many values are present.

In [ ]:
mech.info()

<class 'pandas.DataFrame'>
RangeIndex: 610 entries, 0 to 609
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   action_type          610 non-null    str  
 1   max_phase            610 non-null    int64
 2   mechanism_of_action  610 non-null    str  
 3   molecule_chembl_id   610 non-null    str  
 4   target_chembl_id     610 non-null    str  
dtypes: int64(1), str(4)
memory usage: 24.0 KB


**1.4e — Numerical statistics.** `describe()` summarizes the numeric columns: count, mean, standard deviation, minimum, quartiles and maximum (transposed so that each row is a column).

In [ ]:
describe_numeric(mech)

,count,mean,std,min,25%,50%,75%,max
max_phase,610.0,2.37,1.13,1.0,1.0,2.0,4.0,4.0


**1.4f — Statistics for all columns.** `describe(include="all")` adds the text columns: number of distinct values (`unique`), most frequent value (`top`) and its frequency (`freq`).

In [ ]:
mech.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
action_type,610,3,INHIBITOR,607,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max_phase,610.0,NaN,NaN,NaN,2.372131,1.126284,1.0,1.0,2.0,4.0,4.0
mechanism_of_action,610,102,Tyrosine-protein kinase receptor FLT3 inh...,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
molecule_chembl_id,610,407,CHEMBL3109738,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
target_chembl_id,610,99,CHEMBL1974,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Conclusion 1.4 — mechanisms**
- **Size and types:** 610 rows (one per molecule–target pair) × 5 columns: 1 numeric (`max_phase`, stored as `int64`) and 4 text columns. No missing values.
- **`describe()`:** `max_phase` goes from 1 to 4 with a mean of 2.37, slightly different from the molecules table (2.29) because a molecule with several targets is counted several times.
- **`describe(include="all")`:** 407 distinct molecules act on 99 distinct targets, with 102 distinct mechanism descriptions. `action_type` is almost constant: INHIBITOR in 607 of the 610 rows. The most frequent mechanism is "FLT3 inhibitor" (43 records).

---
## 1.5 Dataset: `targets` — one row per kinase protein (lookup table)

**1.5a — Dimensions and column names.** `shape` gives the number of rows and columns; `columns` lists the variable names.

In [ ]:
print("shape:", tgt.shape, "->", tgt.shape[0], "rows x", tgt.shape[1], "columns")
print("columns:", list(tgt.columns))

shape: (548, 4) -> 548 rows x 4 columns
columns: ['organism', 'pref_name', 'target_chembl_id', 'target_type']


**1.5b — First and last rows.** `head()` and `tail()` show how the data looks and whether the beginning and the end are different (for example sorted).

In [ ]:
print("head()"); display(tgt.head())
print("tail()"); display(tgt.tail())

head()


,organism,pref_name,target_chembl_id,target_type
0,Homo sapiens,Tyrosine-protein kinase ABL1,CHEMBL1862,SINGLE PROTEIN
1,Homo sapiens,Receptor tyrosine-protein kinase erbB-2,CHEMBL1824,SINGLE PROTEIN
2,Homo sapiens,Casein kinase II subunit alpha,CHEMBL3629,SINGLE PROTEIN
3,Homo sapiens,Nucleoside diphosphate kinase A,CHEMBL2159,SINGLE PROTEIN
4,Homo sapiens,Nucleoside diphosphate kinase B,CHEMBL2160,SINGLE PROTEIN


tail()


,organism,pref_name,target_chembl_id,target_type
543,Homo sapiens,FAST kinase domain-containing protein 4,CHEMBL6067265,SINGLE PROTEIN
544,Homo sapiens,Serine/threonine-protein kinase LMTK2,CHEMBL6067300,SINGLE PROTEIN
545,Homo sapiens,Triokinase/FMN cyclase,CHEMBL6067305,SINGLE PROTEIN
546,Homo sapiens,Serine-threonine kinase receptor-associat...,CHEMBL6067330,SINGLE PROTEIN
547,Homo sapiens,Fas-activated serine/threonine kinase,CHEMBL6196088,SINGLE PROTEIN


**1.5c — Random rows.** `sample()` avoids the bias of looking only at the first and last rows (`random_state=42` makes it reproducible).

In [ ]:
tgt.sample(5, random_state=42)

,organism,pref_name,target_chembl_id,target_type
247,Homo sapiens,"Membrane-associated guanylate kinase, WW ...",CHEMBL5212,SINGLE PROTEIN
78,Homo sapiens,Dual specificity mitogen-activated protei...,CHEMBL2964,SINGLE PROTEIN
245,Homo sapiens,MAP/microtubule affinity-regulating kinase 3,CHEMBL5600,SINGLE PROTEIN
353,Homo sapiens,Cyclin-dependent kinase 14,CHEMBL6162,SINGLE PROTEIN
477,Homo sapiens,Serine/threonine-protein kinase VRK3,CHEMBL3430761,SINGLE PROTEIN


**1.5d — Data types and non-null counts.** `info()` gives the pandas type of every column and how many values are present.

In [ ]:
tgt.info()

<class 'pandas.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   organism          548 non-null    str  
 1   pref_name         548 non-null    str  
 2   target_chembl_id  548 non-null    str  
 3   target_type       548 non-null    str  
dtypes: str(4)
memory usage: 17.3 KB


**1.5e — Numerical statistics.** `describe()` summarizes the numeric columns: count, mean, standard deviation, minimum, quartiles and maximum (transposed so that each row is a column).

In [ ]:
describe_numeric(tgt)

No numeric column: describe() has nothing to describe.


**1.5f — Statistics for all columns.** `describe(include="all")` adds the text columns: number of distinct values (`unique`), most frequent value (`top`) and its frequency (`freq`).

In [ ]:
tgt.describe(include="all").T

,count,unique,top,freq
organism,548,1,Homo sapiens,548
pref_name,548,548,Tyrosine-protein kinase ABL1,1
target_chembl_id,548,548,CHEMBL1862,1
target_type,548,1,SINGLE PROTEIN,548


**Conclusion 1.5 — targets**
- **Size and types:** 548 rows × 4 columns, all of them text. There is **no numeric column**, so `describe()` has nothing to show (our helper reports it) and only `describe(include="all")` is informative.
- No missing values. `organism` ("Homo sapiens") and `target_type` ("SINGLE PROTEIN") are constant, and `target_chembl_id` and `pref_name` are unique (548 distinct values each).
- Only a subset of these kinases is actually used elsewhere: 381 appear in `activities` and 99 in `mechanisms` (coverage is analyzed later).

---
## 1.6 Step 1 — Summary across the five datasets

In [ ]:
rows = []
for name, d in files.items():
    n_num = d.select_dtypes("number").shape[1]
    rows.append({"dataset": name, "rows": d.shape[0], "columns": d.shape[1],
                 "numeric columns": n_num, "text columns": d.shape[1] - n_num,
                 "cells": d.size, "missing cells": int(d.isna().sum().sum()),
                 "% missing": round(100 * d.isna().sum().sum() / d.size, 1),
                 "memory (MB)": round(d.memory_usage(deep=True).sum() / 1024**2, 2)})
pd.DataFrame(rows).set_index("dataset")

,rows,columns,numeric columns,text columns,cells,missing cells,% missing,memory (MB)
dataset,,,,,,,,
molecules,407,15,10,5,6105,953,15.6,0.17
activities,15655,11,3,8,172205,675,0.4,7.46
indications,2864,4,1,3,11456,0,0.0,0.56
mechanisms,610,5,1,4,3050,0,0.0,0.16
targets,548,4,0,4,2192,0,0.0,0.14


**Conclusion — Step 1**
- The five datasets have different units of observation (compound, measurement, molecule–disease pair, molecule–target pair, kinase), so they must always be joined on the right key (`molecule_chembl_id` or `target_chembl_id`).
- `activities` is by far the largest table (15 655 rows, about 88% of the memory); the whole dataset is only about 8.5 MB.
- Missing values exist in **only two tables**: `molecules` (about 16% of its cells, concentrated in the structure and descriptor columns and in `first_approval`) and `activities` (0.4%, only in the raw `units` column). `indications`, `mechanisms` and `targets` are complete.
- Several columns are constant or nearly constant (`targets.organism`, `targets.target_type`, `activities.standard_units`, `mechanisms.action_type`) and some are raw duplicates; Step 2 documents what each column is for.

---
# Step 2 — Understand the columns (data dictionary)
For every column of every dataset we record:

| Field | Meaning |
|---|---|
| `column` | variable name |
| `meaning` | what the column represents |
| `pandas_dtype` | type stored by pandas (computed). Text is reported as `object` (called `str` in pandas ≥ 3) |
| `semantic_type` | what the data *is*: identifier, text label, categorical, ordinal, numerical (continuous or count), constant… |
| `encoding` | how the values are represented or stored |
| `n_unique`, `n_missing`, `pct_missing`, `example` | computed from the data |
| `project_role` | how the column is used in DrugScope AI |
| `role_group` | the role summarized in one word, to count and filter: *Clustering input*, *Join key*, *Metadata / filter*, *Label*, *Biological annotation*, *Quality control* or *Drop* |

The meaning, semantic type, encoding and role are written by hand (they need domain knowledge); the type, unique, missing and example columns are computed, so they stay correct if the data changes.

In [ ]:
meta = {
 "molecules": {
  "molecule_chembl_id": ("Unique ChEMBL identifier of the compound", "Identifier", "Text: 'CHEMBL' + number (e.g. CHEMBL9960)", "Join key to the other four files; not a feature", "Join key"),
  "name": ("Drug name or development code of the compound", "Text label", "Upper-case text: drug name or code (e.g. AZD-7624)", "Display label in the application; not a feature", "Label"),
  "max_phase": ("Furthest clinical development phase reached", "Ordinal", "Whole numbers stored as float: 1, 2, 3 = clinical trial phases, 4 = approved", "Metadata: used to derive the 'approved' flag and to filter / interpret clusters; not a clustering feature", "Metadata / filter"),
  "first_approval": ("Year the drug was first approved", "Numerical (year)", "Year stored as float (e.g. 2012.0); empty when the molecule is not approved", "Metadata; not a feature", "Metadata / filter"),
  "molecule_type": ("Kind of molecule", "Categorical (nominal)", "Text with 4 categories: Small molecule, Unknown, Antibody, Protein", "Filter: only 'Small molecule' molecules are clustered", "Metadata / filter"),
  "smiles": ("The molecule's structure written as a line of text (SMILES)", "Text (chemical structure)", "SMILES string; '.' separates fragments (salts, solvents); '@' and '/' encode stereochemistry", "Main input: standardized, then converted to fingerprints and descriptors", "Clustering input"),
  "inchi_key": ("Hashed standard identifier of the structure", "Identifier", "27-character text: connectivity block - stereo/isotope block - protonation letter", "Duplicate detection; not a feature", "Quality control"),
  "mol_weight": ("Molecular weight (size of the molecule)", "Numerical (continuous)", "Float, in daltons; the ChEMBL value includes counter-ions", "Clustering descriptor (recomputed on the parent structure)", "Clustering input"),
  "alogp": ("Calculated lipophilicity, LogP (fat vs water solubility)", "Numerical (continuous)", "Float, unitless (logarithmic scale)", "Clustering descriptor (recomputed)", "Clustering input"),
  "hbd": ("Number of hydrogen-bond donors", "Numerical (discrete count)", "Whole number stored as float", "Clustering descriptor (recomputed)", "Clustering input"),
  "hba": ("Number of hydrogen-bond acceptors", "Numerical (discrete count)", "Whole number stored as float", "Clustering descriptor (recomputed)", "Clustering input"),
  "psa": ("Polar surface area (polar part of the molecule)", "Numerical (continuous)", "Float, in square angstroms", "Clustering descriptor, called TPSA in the project (recomputed)", "Clustering input"),
  "rtb": ("Number of rotatable bonds (flexibility)", "Numerical (discrete count)", "Whole number stored as float", "Clustering descriptor (recomputed)", "Clustering input"),
  "aromatic_rings": ("Number of aromatic rings", "Numerical (discrete count)", "Whole number stored as float", "Extra clustering descriptor (not in the project's list of six)", "Clustering input"),
  "ro5_violations": ("Number of Lipinski drug-likeness rules broken", "Ordinal (derived count)", "Whole number 0-3 stored as float", "Not used: derived from molecular weight, logP, HBD and HBA, so redundant", "Drop"),
 },
 "activities": {
  "assay_chembl_id": ("ID of the experiment (assay) that produced the measurement", "Identifier", "Text: 'CHEMBL' + number", "Used to spot repeated records; not a feature", "Quality control"),
  "molecule_chembl_id": ("ID of the molecule tested", "Identifier", "Text: 'CHEMBL' + number", "Join key to molecules", "Join key"),
  "pchembl_value": ("Potency on a log scale: -log10 of the molar IC50, Ki or Kd (higher = more potent)", "Numerical (continuous)", "Float between 4.0 and 10.8 (e.g. 8.9 is about 1.3 nM)", "Main potency measure for the biological interpretation of the clusters (after aggregation)", "Biological annotation"),
  "standard_type": ("Type of measurement", "Categorical (nominal)", "Text with 3 values: IC50, Ki, Kd", "Grouping key: the three types must not be mixed", "Biological annotation"),
  "standard_units": ("Unit of the standardized value", "Constant", "Text 'nM' in every row", "Dropped: no information", "Drop"),
  "standard_value": ("Measured value converted to nanomolar", "Numerical (continuous, log-distributed)", "Float in nM, from 0.02 to 100 000", "Backup potency measure; analyzed on a log scale", "Biological annotation"),
  "target_chembl_id": ("ID of the kinase tested", "Identifier", "Text: 'CHEMBL' + number", "Join key to targets", "Join key"),
  "target_pref_name": ("Name of the kinase tested", "Text label (categorical)", "Text: preferred protein name", "Readable label for the biological explorer; redundant with target_chembl_id", "Label"),
  "type": ("Measurement type as originally reported", "Categorical (raw)", "Text with 11 variants (e.g. Kd, IC50, pKi)", "Not used: raw copy of standard_type", "Drop"),
  "units": ("Unit as originally reported", "Categorical (raw)", "Text with 15 spellings; empty for measurements reported as p-values", "Not used: raw copy of standard_units", "Drop"),
  "value": ("Value as originally reported", "Numerical (raw, mixed units)", "Float in the original unit (nM, uM... or a p-value)", "Not used: raw copy of standard_value with mixed units", "Drop"),
 },
 "indications": {
  "molecule_chembl_id": ("ID of the molecule", "Identifier", "Text: 'CHEMBL' + number", "Join key to molecules", "Join key"),
  "efo_term": ("Disease or condition the molecule is used for (EFO vocabulary)", "Categorical (nominal, many values)", "Lower-case text, 495 distinct terms (e.g. 'neoplasm')", "Disease annotation for the biological interpretation; to be grouped into broad classes", "Biological annotation"),
  "mesh_heading": ("The same disease in the MeSH vocabulary", "Categorical (nominal, many values)", "Text, 362 distinct headings (e.g. 'Neoplasms')", "Second disease annotation; used to group diseases (oncology vs other)", "Biological annotation"),
  "max_phase_for_ind": ("Furthest stage reached for this specific disease", "Ordinal", "Float: 0.5, 1, 2, 3 or 4 (4 = approved)", "Separates approved indications (4) from investigational ones; defines a 'known indication'", "Biological annotation"),
 },
 "mechanisms": {
  "molecule_chembl_id": ("ID of the molecule", "Identifier", "Text: 'CHEMBL' + number", "Join key to molecules", "Join key"),
  "target_chembl_id": ("ID of the kinase the molecule acts on", "Identifier", "Text: 'CHEMBL' + number", "Join key to targets", "Join key"),
  "action_type": ("How the molecule acts on the target", "Categorical (nominal)", "Text with 3 values: INHIBITOR, NEGATIVE ALLOSTERIC MODULATOR, ANTAGONIST", "Annotation: do molecules of a cluster act the same way?", "Biological annotation"),
  "mechanism_of_action": ("Short text describing the mechanism", "Text (categorical, repeated phrases)", "Text, 102 distinct phrases (e.g. '... kinase inhibitor')", "Readable description in the biological explorer", "Biological annotation"),
  "max_phase": ("Furthest clinical phase of the molecule (copy of the molecules table)", "Ordinal", "Integer 1-4 (int64)", "Not used: repeats molecules.max_phase", "Drop"),
 },
 "targets": {
  "target_chembl_id": ("Unique ChEMBL identifier of the kinase", "Identifier", "Text: 'CHEMBL' + number", "Join key", "Join key"),
  "pref_name": ("Name of the kinase", "Text label", "Text: preferred protein name", "Readable label for the biological explorer", "Label"),
  "organism": ("Species of the target", "Constant", "Text 'Homo sapiens' in every row", "Dropped: no information", "Drop"),
  "target_type": ("Kind of target", "Constant", "Text 'SINGLE PROTEIN' in every row", "Dropped: no information", "Drop"),
 },
}

def norm_dtype(t):
    return "object" if str(t) in ("str", "string", "object") else str(t)

def build_dictionary(df, m):
    assert set(m) == set(df.columns), f"columns without definition: {set(df.columns) ^ set(m)}"
    rows = []
    for c in df.columns:
        meaning, sem, enc, role, group = m[c]
        s = df[c]
        rows.append({
            "column": c, "meaning": meaning, "pandas_dtype": norm_dtype(s.dtype),
            "semantic_type": sem, "encoding": enc,
            "n_unique": int(s.nunique()), "n_missing": int(s.isna().sum()),
            "pct_missing": round(100 * s.isna().mean(), 1),
            "example": str(s.dropna().iloc[0])[:28] if s.notna().any() else None,
            "project_role": role, "role_group": group})
    return pd.DataFrame(rows)

DICT_DIR = RAW.parent / "dictionary"
DICT_DIR.mkdir(exist_ok=True)
dd = {name: build_dictionary(df, meta[name]) for name, df in files.items()}
for name, d in dd.items():
    d.to_csv(DICT_DIR / f"{name}_dictionary.csv", index=False, encoding="utf-8-sig")
print("saved in", DICT_DIR, ":", sorted(p.name for p in DICT_DIR.glob("*.csv")))

saved in /Users/kherrazmaroua/Downloads/ML_mini_project/DrugScrope-ML-main/data/dictionary : ['activities_dictionary.csv', 'indications_dictionary.csv', 'mechanisms_dictionary.csv', 'molecules_dictionary.csv', 'targets_dictionary.csv']


**Conclusion — setup of Step 2.** The `assert` in `build_dictionary` stops the notebook if a column of a dataset has no definition (or if a definition refers to a column that no longer exists), so the dictionaries always cover exactly the columns of the data. Five CSV files were saved, one per dataset.

---
## 2.1 Data dictionary: `molecules`

The dictionary of `molecules`: the first columns are written by hand, `pandas_dtype`, `n_unique`, `n_missing`, `pct_missing` and `example` are computed from the data.

In [ ]:
dd["molecules"]

,column,meaning,pandas_dtype,semantic_type,encoding,n_unique,n_missing,pct_missing,example,project_role,role_group
0,molecule_chembl_id,Unique ChEMBL identifier of the compound,object,Identifier,Text: 'CHEMBL' + number (e.g. CHEMBL9960),407,0,0.0,CHEMBL9960,Join key to the other four files; not a f...,Join key
1,name,Drug name or development code of the comp...,object,Text label,Upper-case text: drug name or code (e.g. ...,407,0,0.0,AZD-7624,Display label in the application; not a f...,Label
2,max_phase,Furthest clinical development phase reached,float64,Ordinal,"Whole numbers stored as float: 1, 2, 3 = ...",4,0,0.0,2.0,Metadata: used to derive the 'approved' f...,Metadata / filter
3,first_approval,Year the drug was first approved,float64,Numerical (year),Year stored as float (e.g. 2012.0); empty...,21,319,78.4,2012.0,Metadata; not a feature,Metadata / filter
4,molecule_type,Kind of molecule,object,Categorical (nominal),"Text with 4 categories: Small molecule, U...",4,0,0.0,Small molecule,Filter: only 'Small molecule' molecules a...,Metadata / filter
5,smiles,The molecule's structure written as a lin...,object,Text (chemical structure),SMILES string; '.' separates fragments (s...,345,62,15.2,CC[C@H](NC(=O)c1cc(-c2ccccc2,"Main input: standardized, then converted ...",Clustering input
6,inchi_key,Hashed standard identifier of the structure,object,Identifier,27-character text: connectivity block - s...,345,62,15.2,MXNGYQJJYRVGGJ-QFIPXVFZSA-N,Duplicate detection; not a feature,Quality control
7,mol_weight,Molecular weight (size of the molecule),float64,Numerical (continuous),"Float, in daltons; the ChEMBL value inclu...",337,62,15.2,366.46,Clustering descriptor (recomputed on the ...,Clustering input
8,alogp,"Calculated lipophilicity, LogP (fat vs wa...",float64,Numerical (continuous),"Float, unitless (logarithmic scale)",239,64,15.7,5.78,Clustering descriptor (recomputed),Clustering input
9,hbd,Number of hydrogen-bond donors,float64,Numerical (discrete count),Whole number stored as float,8,64,15.7,1.0,Clustering descriptor (recomputed),Clustering input


**Conclusion 2.1 — molecules**
- 15 columns: **8 clustering inputs** (`smiles` plus the 7 descriptors `mol_weight`, `alogp`, `hbd`, `hba`, `psa`, `rtb`, `aromatic_rings`), 3 metadata/filter columns (`max_phase`, `first_approval`, `molecule_type`), 1 join key, 1 label, 1 quality-control column (`inchi_key`) and 1 column to drop (`ro5_violations`, derived from other descriptors).
- Missing values: `first_approval` 78.4% (expected: only approved drugs have a year); `smiles`, `inchi_key` and `mol_weight` 15.2%; the other descriptors 15.7%.
- Semantic types: the descriptors are numeric but of two kinds: continuous (`mol_weight`, `alogp`, `psa`) and discrete counts (`hbd`, `hba`, `rtb`, `aromatic_rings`), although all are stored as `float64`. `max_phase` is ordinal despite being stored as a float.

---
## 2.2 Data dictionary: `activities`

The dictionary of `activities`: the first columns are written by hand, `pandas_dtype`, `n_unique`, `n_missing`, `pct_missing` and `example` are computed from the data.

In [ ]:
dd["activities"]

,column,meaning,pandas_dtype,semantic_type,encoding,n_unique,n_missing,pct_missing,example,project_role,role_group
0,assay_chembl_id,ID of the experiment (assay) that produce...,object,Identifier,Text: 'CHEMBL' + number,6679,0,0.0,CHEMBL882727,Used to spot repeated records; not a feature,Quality control
1,molecule_chembl_id,ID of the molecule tested,object,Identifier,Text: 'CHEMBL' + number,262,0,0.0,CHEMBL288441,Join key to molecules,Join key
2,pchembl_value,Potency on a log scale: -log10 of the mol...,float64,Numerical (continuous),Float between 4.0 and 10.8 (e.g. 8.9 is a...,577,0,0.0,8.89,Main potency measure for the biological i...,Biological annotation
3,standard_type,Type of measurement,object,Categorical (nominal),"Text with 3 values: IC50, Ki, Kd",3,0,0.0,IC50,Grouping key: the three types must not be...,Biological annotation
4,standard_units,Unit of the standardized value,object,Constant,Text 'nM' in every row,1,0,0.0,nM,Dropped: no information,Drop
5,standard_value,Measured value converted to nanomolar,float64,"Numerical (continuous, log-distributed)","Float in nM, from 0.02 to 100 000",2824,0,0.0,1.3,Backup potency measure; analyzed on a log...,Biological annotation
6,target_chembl_id,ID of the kinase tested,object,Identifier,Text: 'CHEMBL' + number,381,0,0.0,CHEMBL267,Join key to targets,Join key
7,target_pref_name,Name of the kinase tested,object,Text label (categorical),Text: preferred protein name,381,0,0.0,Proto-oncogene tyrosine-prot,Readable label for the biological explore...,Label
8,type,Measurement type as originally reported,object,Categorical (raw),"Text with 11 variants (e.g. Kd, IC50, pKi)",11,0,0.0,IC50,Not used: raw copy of standard_type,Drop
9,units,Unit as originally reported,object,Categorical (raw),Text with 15 spellings; empty for measure...,15,675,4.3,nM,Not used: raw copy of standard_units,Drop


**Conclusion 2.2 — activities**
- 11 columns: 3 biological annotations (`pchembl_value`, `standard_type`, `standard_value`), 2 join keys, 1 quality-control ID (`assay_chembl_id`), 1 label and **4 columns to drop** (`standard_units`, `type`, `units`, `value`: constant or raw copies).
- Missing values: only `units` (4.3%), and these are the measurements reported as p-values; the standardized columns are complete.
- Semantic types: `standard_value` is numeric but log-distributed, so `pchembl_value` (already on a log scale) is the better potency variable.

---
## 2.3 Data dictionary: `indications`

The dictionary of `indications`: the first columns are written by hand, `pandas_dtype`, `n_unique`, `n_missing`, `pct_missing` and `example` are computed from the data.

In [ ]:
dd["indications"]

,column,meaning,pandas_dtype,semantic_type,encoding,n_unique,n_missing,pct_missing,example,project_role,role_group
0,efo_term,Disease or condition the molecule is used...,object,"Categorical (nominal, many values)","Lower-case text, 495 distinct terms (e.g....",495,0,0.0,non-small cell lung carcinom,Disease annotation for the biological int...,Biological annotation
1,max_phase_for_ind,Furthest stage reached for this specific ...,float64,Ordinal,"Float: 0.5, 1, 2, 3 or 4 (4 = approved)",5,0,0.0,4.0,Separates approved indications (4) from i...,Biological annotation
2,mesh_heading,The same disease in the MeSH vocabulary,object,"Categorical (nominal, many values)","Text, 362 distinct headings (e.g. 'Neopla...",362,0,0.0,"Carcinoma, Non-Small-Cell Lu",Second disease annotation; used to group ...,Biological annotation
3,molecule_chembl_id,ID of the molecule,object,Identifier,Text: 'CHEMBL' + number,359,0,0.0,CHEMBL2105712,Join key to molecules,Join key


**Conclusion 2.3 — indications**
- 4 columns: 1 join key and **3 biological annotations**. No missing values.
- `efo_term` and `mesh_heading` are categorical with many distinct values (495 and 362), so they will need to be grouped into broader disease classes. `max_phase_for_ind` is ordinal and is what separates an approved indication (4) from an investigational one.

---
## 2.4 Data dictionary: `mechanisms`

Before documenting `max_phase` as redundant, we verify it: does the phase recorded in `mechanisms` equal the phase of the same molecule in `molecules`?

In [ ]:
chk = (mech[["molecule_chembl_id", "max_phase"]].drop_duplicates()
       .merge(mol[["molecule_chembl_id", "max_phase"]], on="molecule_chembl_id", suffixes=("_mech", "_mol")))
print(len(chk), "molecules compared | identical everywhere:", (chk.max_phase_mech == chk.max_phase_mol).all())

407 molecules compared | identical everywhere: True


The dictionary of `mechanisms`: the first columns are written by hand, `pandas_dtype`, `n_unique`, `n_missing`, `pct_missing` and `example` are computed from the data.

In [ ]:
dd["mechanisms"]

,column,meaning,pandas_dtype,semantic_type,encoding,n_unique,n_missing,pct_missing,example,project_role,role_group
0,action_type,How the molecule acts on the target,object,Categorical (nominal),"Text with 3 values: INHIBITOR, NEGATIVE A...",3,0,0.0,INHIBITOR,Annotation: do molecules of a cluster act...,Biological annotation
1,max_phase,Furthest clinical phase of the molecule (...,int64,Ordinal,Integer 1-4 (int64),4,0,0.0,4,Not used: repeats molecules.max_phase,Drop
2,mechanism_of_action,Short text describing the mechanism,object,"Text (categorical, repeated phrases)","Text, 102 distinct phrases (e.g. '... kin...",102,0,0.0,Receptor protein-tyrosine ki,Readable description in the biological ex...,Biological annotation
3,molecule_chembl_id,ID of the molecule,object,Identifier,Text: 'CHEMBL' + number,407,0,0.0,CHEMBL1201585,Join key to molecules,Join key
4,target_chembl_id,ID of the kinase the molecule acts on,object,Identifier,Text: 'CHEMBL' + number,99,0,0.0,CHEMBL1824,Join key to targets,Join key


**Conclusion 2.4 — mechanisms**
- 5 columns: 2 join keys, 2 biological annotations (`action_type`, `mechanism_of_action`) and 1 column to drop (`max_phase`). No missing values.
- The check above confirms that `max_phase` is an exact copy of the value in `molecules`, so it carries no new information.

---
## 2.5 Data dictionary: `targets`

The dictionary of `targets`: the first columns are written by hand, `pandas_dtype`, `n_unique`, `n_missing`, `pct_missing` and `example` are computed from the data.

In [ ]:
dd["targets"]

,column,meaning,pandas_dtype,semantic_type,encoding,n_unique,n_missing,pct_missing,example,project_role,role_group
0,organism,Species of the target,object,Constant,Text 'Homo sapiens' in every row,1,0,0.0,Homo sapiens,Dropped: no information,Drop
1,pref_name,Name of the kinase,object,Text label,Text: preferred protein name,548,0,0.0,Tyrosine-protein kinase ABL1,Readable label for the biological explorer,Label
2,target_chembl_id,Unique ChEMBL identifier of the kinase,object,Identifier,Text: 'CHEMBL' + number,548,0,0.0,CHEMBL1862,Join key,Join key
3,target_type,Kind of target,object,Constant,Text 'SINGLE PROTEIN' in every row,1,0,0.0,SINGLE PROTEIN,Dropped: no information,Drop


**Conclusion 2.5 — targets**
- 4 columns: 1 join key, 1 label and **2 constant columns to drop** (`organism`, `target_type`). No missing values.

---
## 2.6 Step 2 — Summary: the role of the 39 columns

In [ ]:
all_dd = pd.concat({n: d for n, d in dd.items()}, names=["dataset", "i"]).reset_index("dataset")
role_table = all_dd.pivot_table(index="role_group", columns="dataset", values="column", aggfunc="count", fill_value=0)
role_table = role_table[list(files)]
role_table["total"] = role_table.sum(axis=1)
role_table.loc["total"] = role_table.sum()
role_table

dataset,molecules,activities,indications,mechanisms,targets,total
role_group,,,,,,
Biological annotation,0,3,3,2,0,8
Clustering input,8,0,0,0,0,8
Drop,1,4,0,1,2,8
Join key,1,2,1,2,1,7
Label,1,1,0,0,1,3
Metadata / filter,3,0,0,0,0,3
Quality control,1,1,0,0,0,2
total,15,11,4,5,4,39


In [ ]:
print("Columns with missing values:")
all_dd[all_dd.n_missing > 0][["dataset", "column", "n_missing", "pct_missing"]].reset_index(drop=True)

Columns with missing values:


,dataset,column,n_missing,pct_missing
0,molecules,first_approval,319,78.4
1,molecules,smiles,62,15.2
2,molecules,inchi_key,62,15.2
3,molecules,mol_weight,62,15.2
4,molecules,alogp,64,15.7
5,molecules,hbd,64,15.7
6,molecules,hba,64,15.7
7,molecules,psa,64,15.7
8,molecules,rtb,64,15.7
9,molecules,aromatic_rings,64,15.7


**Conclusion — Step 2**
- The five datasets have **39 columns**: 8 clustering inputs (all in `molecules`), 7 join keys, 3 metadata/filter columns, 3 labels, 8 biological annotations, 2 quality-control columns and **8 columns to drop** (constant, redundant or raw duplicates).
- Only **8 of the 39 columns** feed the clustering; everything else either links the files, filters the molecules, or is kept for the independent biological interpretation, as required by the project design.
- Missing values appear in **12 columns**: 11 in `molecules` and the raw `units` column in `activities`; the other three datasets are complete.
- Drop list: `molecules.ro5_violations`, `activities.standard_units`, `activities.type`, `activities.units`, `activities.value`, `mechanisms.max_phase`, `targets.organism`, `targets.target_type`.
- The dictionaries are saved in `data/dictionary/` as `<dataset>_dictionary.csv` and can be included directly in the report.

*Next: the data-quality step (missing values, duplicates, salts and invalid structures).*